# ETL Transformación - De Staging a DWH

Este notebook realiza la transformación de datos desde las tablas staging (STG_Universidad) hacia el Data Warehouse (dw_universidad).

## Procesos incluidos:
- **Limpieza**: Reparación de encoding, espacios, valores nulos
- **Validación**: Tipos de datos, rangos, integridad referencial
- **Normalización**: Formatos de fechas, mayúsculas, decimales
- **Deduplicación**: Eliminación de registros duplicados
- **Optimización**: Procesamiento por lotes y transacciones

In [ ]:
# ╔════════════════════════════════════════════════════════════════════════════╗
# ║         CELDA 2: SETUP - IMPORTACIONES Y CONEXIONES                        ║
# ╚════════════════════════════════════════════════════════════════════════════╝

# PROPÓSITO:
# Configurar el entorno del notebook de TRANSFORMACIÓN:
# - Importar librerías necesarias
# - Conectar a TWO bases de datos (STG para lectura, DWH para escritura)
# - Inicializar logger
# 
# ARQUITECTURA:
# STG_Universidad (STAGING) ──> TRANSFORM ──> dw_universidad (DWH)
#     Datos crudos                Limpiar      Dimensional

# ═════════════════════════════════════════════════════════════════════════════

# IMPORTS ESTÁNDAR
import pandas as pd                    # DataFrames
import numpy as np                     # Arrays numéricos
from sqlalchemy import create_engine, text, MetaData, Table  # ORM SQL
from sqlalchemy.pool import NullPool   # Pool de conexiones
from datetime import datetime, date    # Manejo de fechas
from dotenv import load_dotenv         # Variables de entorno
import os                              # Sistema operativo
from pathlib import Path               # Rutas modernas
import warnings                        # Control de advertencias
import re                              # Expresiones regulares
from typing import Tuple, Dict, List   # Type hints

# Suprimir advertencias innecesarias
warnings.filterwarnings('ignore')

# Agregar directorio padre (TP2/) al path
sys.path.append(os.path.join(os.getcwd(), '..'))

# Importar LoggerManager centralizado
from logging_config import LoggerManager

# ─ CONFIGURACIÓN: CREDENCIALES Y CONEXIONES ─────────────────────────────────

# Cargar variables de entorno desde .env
load_dotenv()

# Obtener credenciales
USER = os.getenv("DB_USER")
PASSWORD = os.getenv("DB_PASSWORD")
HOST = os.getenv("DB_HOST")
PORT = os.getenv("DB_PORT")
STG_DATABASE = os.getenv("STG_DATABASE")      # Base STAGING (lectura)
DWH_DATABASE = os.getenv("DWH_DATABASE")      # Base DWH (escritura)

# Crear MOTOR #1: Staging (lectura de datos crudos)
# NullPool: no cachea conexiones (mejor para batch processing)
engine_stg = create_engine(
    f"mysql+pymysql://{USER}:{PASSWORD}@{HOST}:{PORT}/{STG_DATABASE}",
    poolclass=NullPool
)

# Crear MOTOR #2: DWH (escritura de datos transformados)
engine_dwh = create_engine(
    f"mysql+pymysql://{USER}:{PASSWORD}@{HOST}:{PORT}/{DWH_DATABASE}",
    poolclass=NullPool
)

print("\n📊 TRANSFORMACIÓN DIMENSIONAL")
print("="*80)
print("[✓] Motores de conexión configurados:")
print(f"    → Lectura:  {STG_DATABASE} en {HOST}:{PORT}")
print(f"    → Escritura: {DWH_DATABASE} en {HOST}:{PORT}")

# ─ CONFIGURAR LOGGING ────────────────────────────────────────────────────────
logger = LoggerManager.configurar(
    "transformacion",
    ruta_raiz=os.getcwd(),
    carpeta_logs='logs'
)

# Obtener ruta de logs
RUTA_LOGS = LoggerManager.obtener_ruta_logs()

print(f"[✓] Logging configurado en: {RUTA_LOGS}")

2026-05-02 00:20:08,191 - INFO - Iniciando transformación. Log: c:\Users\maria\OneDrive\Documentos\GitHub\ADE2026_TpiUniversidad\TP2\ETL_Transformacion\logs\transformacion_20260502_002008.log


[INFO] Motores de conexión configurados
  Staging: None en localhost:3306
  DWH: None en localhost:3306

[OK] Logging configurado en: c:\Users\maria\OneDrive\Documentos\GitHub\ADE2026_TpiUniversidad\TP2\ETL_Transformacion\logs\transformacion_20260502_002008.log


In [ ]:
# ╔════════════════════════════════════════════════════════════════════════════╗
# ║    CELDA 3: CLASE DataCleaner - FUNCIONES DE LIMPIEZA Y VALIDACIÓN         ║
# ╚════════════════════════════════════════════════════════════════════════════╝

# PROPÓSITO:
# Clase auxiliar con métodos estáticos para limpiar y validar datos crudos
# de STAGING antes de cargar en DWH.
#
# RESPONSABILIDADES:
# 1. Limpiar strings: espacios, encoding, valores nulos
# 2. Validar números: rango, formato
# 3. Parsear fechas: múltiples formatos
# 4. Normalizar categorías: género, períodos, estados
# 5. Validar reglas de negocio: DNI argentino, notas 0-10

# ═════════════════════════════════════════════════════════════════════════════

class DataCleaner:
    """
    ════════════════════════════════════════════════════════════════════════════
    CLASE: DataCleaner
    ════════════════════════════════════════════════════════════════════════════
    
    Métodos estáticos para limpieza y validación de datos.
    
    PROPÓSITO: Centralizar lógica de limpieza para reutilizar en múltiples
    transformaciones sin duplicar código.
    
    EJEMPLO DE USO:
        cleaner = DataCleaner()
        nombre_limpio = cleaner.limpiar_string(nombre_sucio)
        edad = cleaner.limpiar_numero("25,5", tipo="int")
        fecha = cleaner.limpiar_fecha("15/05/2023")
        validado = cleaner.validar_dni("35678901")
    """
    
    @staticmethod
    def limpiar_string(valor: str) -> str:
        """
        INPUT: valor (str|None): String potencialmente sucio o null
        OUTPUT: str|None: String limpio o None si es nulo/vacío
        
        TRATAMIENTO:
        1. Si es None/NaN → retorna None
        2. Convertir a string y trim()
        3. Si es valor nulo textual ('null', 'N/A', 'sin dato') → None
        4. Reparar encoding (mojibake UTF-8/Latin-1)
        5. Retornar texto limpio
        
        EJEMPLO:
            " Juan Pérez  " → "Juan Pérez"
            "null" → None
            "" → None
        """
        if pd.isna(valor) or valor is None:
            return None
        
        # Convertir a string si no lo es
        valor = str(valor).strip()
        
        # Valores nulos textuales
        if valor.lower() in ['', 'null', 'none', 'n/a', 'sin dato']:
            return None
        
        # Reparar encoding corrupido (caracteresÃ¡ en vez de á)
        try:
            valor = valor.encode('latin-1').decode('utf-8', errors='ignore')
        except:
            pass
        
        return valor
    
    @staticmethod
    def limpiar_numero(valor, tipo='float', requerido=False) -> any:
        """
        INPUT: 
        - valor: número potencialmente sucio ("25,5", " 100", None, "null")
        - tipo: "int" o "float"
        - requerido: si True y está vacío, registra warning
        
        OUTPUT: int|float|None
        
        TRATAMIENTO:
        1. Si es None/vacío → retorna None (o warning si requerido)
        2. Convertir string → número
        3. Manejar separadores comunes: coma → punto
        4. Validar rango según tipo
        5. Retornar número convertido
        
        EJEMPLO:
            "25,5" (tipo="float") → 25.5
            " 100 " (tipo="int") → 100
            "abc" (tipo="int") → None (con warning)
        """
        if pd.isna(valor) or valor is None:
            if requerido:
                LoggerManager.warning("Valor requerido faltante")
            return None
        
        valor_str = str(valor).strip()
        
        if valor_str.lower() in ['', 'null', 'none', 'n/a']:
            return None
        
        try:
            # Remover espacios y reemplazar coma por punto
            valor_str = valor_str.replace(',', '.').replace(' ', '')
            
            if tipo == 'int':
                return int(float(valor_str))
            elif tipo == 'float':
                return float(valor_str)
            else:
                return None
        except:
            LoggerManager.warning(f"No se pudo convertir '{valor}' a {tipo}")
            return None
    
    @staticmethod
    def limpiar_fecha(valor) -> date:
        """
        INPUT: valor (str|int|date): Fecha en múltiples formatos posibles
        OUTPUT: date|None
        
        TRATAMIENTO:
        1. Probar formatos comunes (ISO, latino, compacto, etc)
        2. Si ninguno funciona: intentar parseo genérico
        3. Si todavía falla: registrar warning y retornar None
        
        FORMATOS SOPORTADOS:
        - 2023-05-15 (ISO)
        - 15/05/2023 (Latino)
        - 20230515 (Compacto)
        - 15-05-2023 (Latino guión)
        - 05-15-2023 (Americano)
        - 2023 (Solo año → 2023-01-01)
        
        EJEMPLO:
            "15/05/2023" → date(2023,5,15)
            "2023-05-15" → date(2023,5,15)
            "2023" → date(2023,1,1)
            "invalid" → None
        """
        if pd.isna(valor) or valor is None:
            return None
        
        valor_str = str(valor).strip()
        
        if valor_str.lower() in ['', 'null', 'none', 'n/a']:
            return None
        
        # Lista de formatos a intentar (en orden)
        formatos = [
            '%Y-%m-%d',   # ISO 2023-05-15
            '%d/%m/%Y',   # Latino 15/05/2023
            '%Y%m%d',     # Compacto 20230515
            '%d-%m-%Y',   # Latino guión 15-05-2023
            '%m-%d-%Y',   # Americano 05-15-2023
            '%Y',         # Solo año 2023
            '%d/%m/%y',   # Latino año corto 15/05/23
            '%Y/%m/%d',   # ISO barra 2023/05/15
        ]
        
        # Probar cada formato
        for fmt in formatos:
            try:
                return pd.to_datetime(valor_str, format=fmt).date()
            except:
                continue
        
        # Intento final sin formato específico (pandas adivina)
        try:
            return pd.to_datetime(valor_str, dayfirst=True).date()
        except:
            LoggerManager.warning(f"No se pudo parsear fecha: '{valor}'")
            return None
    
    @staticmethod
    def limpiar_genero(valor: str) -> str:
        """
        INPUT: valor (str): Valor de género en diferentes formas
        OUTPUT: str|None: Normalizado a 'M', 'F', 'X' o None
        
        MAPA DE NORMALIZACIÓN:
        - 'M' → 'M' (hombre)
        - 'F' → 'F' (mujer)
        - 'X' → 'X' (otro)
        - 'MASCULINO', 'MALE', 'HOMBRE', '1' → 'M'
        - 'FEMENINO', 'FEMALE', 'MUJER', '2' → 'F'
        - 'OTRO', 'NO BINARIO', 'NB' → 'X'
        
        EJEMPLO:
            "MASCULINO" → "M"
            "f" → "F"
            "another" → None + warning
        """
        if pd.isna(valor) or valor is None:
            return None
        
        valor = str(valor).strip().upper()
        
        if valor in ['M', 'MASCULINO', 'MALE', 'HOMBRE','1']:
            return 'M'
        elif valor in ['F', 'FEMENINO', 'FEMALE', 'MUJER','2']:
            return 'F'
        elif valor in ['X', 'OTRO', 'NO BINARIO', 'NB']:
            return 'X'
        else:
            LoggerManager.warning(f"Género desconocido: '{valor}'")
            return None
    
    @staticmethod
    def validar_dni(dni) -> bool:
        """
        INPUT: dni: Valor potencial de DNI (string, int, None)
        OUTPUT: bool: True si es DNI argentino válido, False en otro caso
        
        REGLA DE NEGOCIO:
        - DNI argentino: 7-8 dígitos
        - Rango: 1,000,000 a 99,999,999
        
        MANEJA:
        - Espacios: " 35 678 901" → 35678901
        - Puntos: "35.678.901" → 35678901
        - Decimales: "35678901.0" → 35678901
        
        EJEMPLO:
            "35678901" → True
            "35.678.901" → True
            "123" → False (muy corto)
            "999999999" → False (muy largo)
            "abc" → False (no numérico)
        """
        if pd.isna(dni) or dni is None:
            return False
        
        try:
            # Limpiar: quitar puntos y espacios
            dni_limpio = str(dni).replace('.', '').replace(' ', '').strip()
            
            # Pasar a entero (maneja decimales como "35678901.0")
            dni_num = int(float(dni_limpio))
            
            # Validar rango: 7-8 dígitos (rango argentino)
            return 1000000 <= dni_num <= 99999999
            
        except (ValueError, TypeError):
            return False
    
    @staticmethod
    def validar_nota(nota: float, minimo: float = 0, maximo: float = 10) -> bool:
        """
        INPUT:
        - nota: valor numérico
        - minimo: valor mínimo válido (default 0)
        - maximo: valor máximo válido (default 10)
        
        OUTPUT: bool
        
        EJEMPLO:
            validar_nota(7.5) → True (en rango 0-10)
            validar_nota(15) → False (mayor a 10)
            validar_nota(-1) → False (menor a 0)
            validar_nota(None) → False
        """
        if pd.isna(nota) or nota is None:
            return False
        
        try:
            nota_num = float(nota)
            return minimo <= nota_num <= maximo
        except:
            return False

print("[✓] Clase DataCleaner cargada")

[OK] Clase DataCleaner cargada


In [ ]:
# ╔════════════════════════════════════════════════════════════════════════════╗
# ║    CELDA 4: FUNCIONES DE TRANSFORMACIÓN POR TABLA                          ║
# ╚════════════════════════════════════════════════════════════════════════════╝

# PROPÓSITO:
# Definir funciones para transformar datos de CADA tabla de staging.
# Cada función:
# 1. Limpia columnas (usando DataCleaner)
# 2. Valida según reglas de negocio
# 3. Separa válidos de inválidos
# 4. Deduplica por clave natural
# 5. Retorna (DataFrame_limpio, estadísticas)
#
# PATRÓN COMÚN:
# - INPUT: DataFrame crudo desde staging (columnas con _raw)
# - OUTPUT: (DataFrame limpio, diccionario de estadísticas)
# - ESTADÍSTICAS: {'total': N, 'válidos': V, 'rechazados': R, 'duplicados': D}
#
# REUTILIZACIÓN:
# Estas funciones se usan en AMBOS procesos:
# - Carga Inicial: transformacion.ipynb (celda 5-7)
# - Carga Incremental: carga_incremental.py (importa como base_etl.transformar_*_base)

# ════════════════════════════════════════════════════════════════════════════

def transformar_estudiante_base(df: pd.DataFrame) -> Tuple[pd.DataFrame, Dict]:
    """
    FUNCIÓN: transformar_estudiante_base(df)
    
    INPUT: DataFrame de stg_estudiante (columnas con _raw)
    OUTPUT: (DataFrame limpio, estadísticas de limpieza)
    
    LIMPIEZA:
    1. id_estudiante: entero positivo (clave)
    2. dni: validar rango argentino (7-8 dígitos)
    3. apellido, nombre: strings limpios
    4. género: normalizar a M/F/X
    5. fecha_nacimiento: parsear múltiples formatos
    6. nacionalidad: string limpio
    7. id_programa: entero (clave foránea)
    8. anio_ingreso: entero (año)
    
    VALIDACIÓN:
    - RECHAZAR si: falta id_estudiante, DNI inválido, faltan apellido/nombre
    - DEDUPLICAR por: DNI (mantener primer registro)
    
    EJEMPLO:
    Input:  {'id_estudiante_raw': '100', 'dni_raw': '35.678.901', ...}
    Output: {'id_estudiante': 100, 'dni': 35678901, ...}
    """

[OK] Funciones de transformación cargadas


In [ ]:
# ╔════════════════════════════════════════════════════════════════════════════╗
# ║   CELDA 5: FUNCIÓN DE INSERCIÓN CON LOTES Y TRANSACCIONES                  ║
# ╚════════════════════════════════════════════════════════════════════════════╝

# PROPÓSITO:
# Insertar datos transformados en tablas DWH de forma eficiente y segura.
# 
# ESTRATEGIA:
# - TRUNCATE: Vacía tabla antes (idempotencia: ejecutar N veces = mismo resultado)
# - LOTES: Divide registro en chunks de 500 para evitar agotamiento de memoria
# - TRANSACCIONES: Integridad - todo o nada
# - MANEJO ERRORES: Falla de 1 lote NO detiene otros
#
# REUTILIZACIÓN:
# Se usa en múltiples transformaciones (dim_tiempo, dim_alumno, fact_inscripcion, etc)

# ════════════════════════════════════════════════════════════════════════════

def insertar_en_lotes(df: pd.DataFrame, tabla_destino: str, tamaño_lote: int = 500) -> Dict:
    """
    ════════════════════════════════════════════════════════════════════════════
    FUNCIÓN: insertar_en_lotes(df, tabla_destino, tamaño_lote)
    ════════════════════════════════════════════════════════════════════════════
    
    INPUT:
    - df: DataFrame con datos transformados listos para DWH
    - tabla_destino: Nombre de tabla DWH donde insertar
    - tamaño_lote: Registros por lote (default 500, más rápido que 1 por 1)
    
    OUTPUT:
    Dict con estadísticas:
    {
        'insertados': cantidad registros insertados,
        'errores': cantidad registros con error,
        'lotes': cantidad lotes procesados
    }
    
    ALGORITMO (7 pasos):
    1. Validar: DataFrame no vacío
    2. Calcular: número de lotes = ceil(N / tamaño_lote)
    3. TRUNCATE: vaciar tabla (idempotencia)
    4. LOOP por cada lote:
        4.1. Extraer slice de lote
        4.2. Insertar con to_sql()
        4.3. Contar registros insertados
        4.4. Si error: registrar y continuar
    5. Retornar estadísticas
    
    TRATAMIENTO DE ERRORES:
    - Si tabla vacía → warnings pero no falla
    - Si un lote falla → registra error, sigue con siguiente
    - Si TRUNCATE falla → error crítico
    
    EJEMPLO:
    >>> df_alumno = {...}  # 2500 estudiantes
    >>> stats = insertar_en_lotes(df_alumno, 'Alumno', tamaño_lote=500)
    >>> print(stats)
    {'insertados': 2500, 'errores': 0, 'lotes': 5}
    # Significado: 5 lotes de 500 registros cada uno, todos bien
    """
    if df.empty:
        LoggerManager.warning(f"DataFrame vacío para {tabla_destino}")
        return {'insertados': 0, 'errores': 0, 'lotes': 0}
    
    resultados = {'insertados': 0, 'errores': 0, 'lotes': 0}
    
    # Paso 1: Calcular número de lotes
    num_lotes = (len(df) + tamaño_lote - 1) // tamaño_lote
    
    try:
        # Paso 2: TRUNCATE tabla DWH (limpia antes de insertar)
        with engine_dwh.begin() as connection:
            connection.execute(text(f"TRUNCATE TABLE {tabla_destino}"))
            LoggerManager.info(f"TRUNCATE: {tabla_destino}")
        
        # Paso 3: Procesar lotes
        for i in range(num_lotes):
            inicio = i * tamaño_lote
            fin = min(inicio + tamaño_lote, len(df))
            lote = df.iloc[inicio:fin].copy()
            
            try:
                # Insertar lote (método multi = múltiples INSERT en 1 statement)
                lote.to_sql(
                    name=tabla_destino,
                    con=engine_dwh,
                    if_exists='append',
                    index=False,
                    method='multi'
                )
                
                registros_lote = len(lote)
                resultados['insertados'] += registros_lote
                resultados['lotes'] += 1
                
                print(f"  Lote {i+1}/{num_lotes}: {registros_lote} registros insertados")
                LoggerManager.info(f"Lote {i+1}/{num_lotes} en {tabla_destino}: {registros_lote} registros")
                
            except Exception as e:
                resultados['errores'] += len(lote)
                LoggerManager.error(f"Error en lote {i+1} de {tabla_destino}: {str(e)}")
                print(f"  [ERROR] Lote {i+1}/{num_lotes}: {str(e)}")
        
    except Exception as e:
        LoggerManager.error(f"Error crítico en inserción de {tabla_destino}: {str(e)}")
        print(f"[ERROR CRÍTICO] {tabla_destino}: {str(e)}")
    
    return resultados

print("[✓] Función de inserción con lotes cargada")

[OK] Función de inserción con lotes cargada


In [ ]:
# ╔════════════════════════════════════════════════════════════════════════════╗
# ║       CELDA 6: ORQUESTACIÓN DE TRANSFORMACIÓN (PROCESAMIENTO PRINCIPAL)     ║
# ╚════════════════════════════════════════════════════════════════════════════╝

# PROPÓSITO:
# ORQUESTAR el flujo ETL completo:
# 1. Para CADA tabla de staging
# 2. Leer → Transformar → Insertar → Validar
# 3. Registrar estadísticas
#
# ORDEN DE PROCESAMIENTO:
# ① Dimensiones estructurales: Facultad → Departamento → Programa → Curso
# ② Dimensiones operacionales: Docente → Estudiante
# ③ Tablas vinculantes: Dictado → Curso_Programa
# ④ Hechos (transaccionales): Inscripción → Examen → EvaluacionCurso
#
# LÓGICA DE ERROR:
# - Si una tabla falla: registra error pero continúa (no detiene todo)
# - Mejor tener 10 tablas que 0 a que falte completar por 1 error

# ════════════════════════════════════════════════════════════════════════════

print("\n" + "="*80)
print("📊 INICIANDO TRANSFORMACIÓN - ETL STAGING → DWH")
print("="*80 + "\n")

# Diccionario que mapea:
# tabla_staging → tabla_dwh → función_transformación
#
# Significado:
# - 1ª columna: tabla SOURCE (de donde leer, en STG_Universidad)
# - 2ª columna: tabla DESTINO (donde escribir, en dw_universidad)
# - 3ª columna: función_transformación (definida arriba) que limpia y valida
transformaciones = [
    ('stg_facultad', 'Facultad', transformar_facultad),
    ('stg_departamento', 'Departamento', transformar_departamento),
    ('stg_programa', 'Programa', transformar_programa),
    ('stg_curso', 'Curso', transformar_curso),
    ('stg_curso_programa', 'CursoPrograma', transformar_curso_programa),
    ('stg_docente', 'Docente', transformar_docente),
    ('stg_estudiante', 'Estudiante', transformar_estudiante),
    ('stg_dictado', 'Dictado', transformar_dictado),
    ('stg_inscripcion', 'Inscripcion', transformar_inscripcion),
    ('stg_examen', 'Examen', transformar_examen),
    ('stg_evaluacion_curso', 'EvaluacionCurso', transformar_evaluacion_curso),
]

# Diccionario para acumular estadísticas de todas las tablas
reporte_general = {}

# ════════════════════════════════════════════════════════════════════════════
# LOOP PRINCIPAL: Procesar cada tabla
# ════════════════════════════════════════════════════════════════════════════

for tabla_stg, tabla_dwh, funcion_transform in transformaciones:
    print(f"\n{'='*80}")
    print(f"🔄 Transformando: {tabla_stg} → {tabla_dwh}")
    print(f"{'='*80}")
    
    try:
        # ════════════════════════════════════════════════════════════════
        # PASO 1: LECTURA desde Staging
        # ════════════════════════════════════════════════════════════════
        # Objetivo: Traer datos CRUDOS a memoria
        # Método: SQL SELECT directamente desde STG
        # Resultado: DataFrame con columnas sufijadas _raw
        
        print(f"\n[1️⃣  LECTURA] Extrayendo datos desde {tabla_stg}...")
        df_staging = pd.read_sql(f"SELECT * FROM {tabla_stg}", con=engine_stg)
        print(f"    → {len(df_staging)} registros leídos desde STAGING")
        LoggerManager.info(f"Lectura {tabla_stg}: {len(df_staging)} registros")
        
        # Si tabla vacía, saltar al siguiente
        if df_staging.empty:
            print(f"    [⚠️  ADVERTENCIA] Tabla {tabla_stg} está vacía")
            LoggerManager.warning(f"Tabla vacía: {tabla_stg}")
            reporte_general[tabla_stg] = {
                'lectura': 0,
                'transformacion': {'total': 0},
                'insercion': {'insertados': 0}
            }
            continue
        
        # ════════════════════════════════════════════════════════════════
        # PASO 2: TRANSFORMACIÓN
        # ════════════════════════════════════════════════════════════════
        # Objetivo: Limpiar y validar datos crudos
        # Método: Llamar función específica para esta tabla
        # Resultado: DataFrame limpio + estadísticas de limpieza
        
        print(f"\n[2️⃣  TRANSFORMACIÓN] Aplicando limpieza y validación...")
        df_transformado, stats_transform = funcion_transform(df_staging)
        
        # Imprimir estadísticas de limpieza
        print(f"    → Total leído:        {stats_transform['total']:6} registros")
        print(f"    → Válidos (aceptados):{stats_transform['válidos']:6} registros")
        print(f"    → Rechazados:         {stats_transform['rechazados']:6} registros")
        print(f"    → Duplicados/removidos:{stats_transform['duplicados']:6} registros")
        print(f"    → SALIDA FINAL:       {len(df_transformado):6} registros listos para DWH")
        
        LoggerManager.info(f"Transformación {tabla_stg}: {len(df_transformado)} registros válidos")
        
        # ════════════════════════════════════════════════════════════════
        # PASO 3: INSERCIÓN en DWH
        # ════════════════════════════════════════════════════════════════
        # Objetivo: Cargar datos transformados en DWH
        # Método: insertar_en_lotes() con TRUNCATE previo
        # Resultado: Estadísticas de inserción
        
        print(f"\n[3️⃣  INSERCIÓN] Cargando en DWH ({tabla_dwh})...")
        stats_insert = insertar_en_lotes(df_transformado, tabla_dwh, tamaño_lote=500)
        
        print(f"    → Insertados: {stats_insert['insertados']:6} registros")
        print(f"    → Errores:    {stats_insert['errores']:6} registros")
        print(f"    → Lotes:      {stats_insert['lotes']:6} lotes procesados")
        
        LoggerManager.info(f"Inserción {tabla_dwh}: {stats_insert['insertados']} registros")
        
        # ════════════════════════════════════════════════════════════════
        # PASO 4: VALIDACIÓN POST-INSERCIÓN
        # ════════════════════════════════════════════════════════════════
        # Objetivo: Verificar integridad después de carga
        # Método: Contar registros en tabla DWH
        # Resultado: ¿Coincide número insertado con número en BD?
        
        print(f"\n[4️⃣  VALIDACIÓN] Verificando integridad en DWH...")
        with engine_dwh.connect() as conn:
            result = conn.execute(text(f"SELECT COUNT(*) FROM {tabla_dwh}"))
            count_final = result.fetchone()[0]
        
        # Comparar: ¿insertados == finales?
        status = "✓ OK" if count_final == stats_insert['insertados'] else "⚠️  WARN"
        print(f"    {status} Registros en {tabla_dwh}: {count_final}")
        
        # ════════════════════════════════════════════════════════════════
        # GUARDAR ESTADÍSTICAS
        # ════════════════════════════════════════════════════════════════
        
        reporte_general[tabla_stg] = {
            'lectura': len(df_staging),
            'transformacion': stats_transform,
            'insercion': stats_insert,
            'final': count_final
        }
        
        print(f"\n[✓] Transformación completada para {tabla_stg}")
        LoggerManager.info(f"COMPLETADO: {tabla_stg} → {tabla_dwh}")
        
    except Exception as e:
        # Capturar error pero CONTINUAR con siguiente tabla
        print(f"\n[❌ ERROR] Fallo en transformación de {tabla_stg}: {str(e)}")
        LoggerManager.error(f"Error en {tabla_stg}: {str(e)}")
        reporte_general[tabla_stg] = {'error': str(e)}

print("\n" + "="*80)
print("✓ TRANSFORMACIÓN COMPLETADA")
print("="*80)


INICIANDO TRANSFORMACIÓN - ETL STAGING -> DWH


Transformando: stg_facultad -> Facultad

[1] Leyendo datos desde stg_facultad...

[ERROR] Fallo en transformación de stg_facultad: (pymysql.err.OperationalError) (1049, "Unknown database 'none'")
(Background on this error at: https://sqlalche.me/e/20/e3q8)


2026-05-02 00:20:09,339 - ERROR - Error en stg_facultad: (pymysql.err.OperationalError) (1049, "Unknown database 'none'")
(Background on this error at: https://sqlalche.me/e/20/e3q8)
Traceback (most recent call last):
  File "c:\Users\maria\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sqlalchemy\engine\base.py", line 143, in __init__
    self._dbapi_connection = engine.raw_connection()
                             ~~~~~~~~~~~~~~~~~~~~~^^
  File "c:\Users\maria\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sqlalchemy\engine\base.py", line 3317, in raw_connection
    return self.pool.connect()
           ~~~~~~~~~~~~~~~~~^^
  File "c:\Users\maria\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sqlalchemy\pool\base.py", line 448, in connect
    return _ConnectionFairy._checkout(self)
           ~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^
  File "c:\Users\maria\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sqlalchemy\pool\base.py", line 1272, in _checkou


Transformando: stg_departamento -> Departamento

[1] Leyendo datos desde stg_departamento...

[ERROR] Fallo en transformación de stg_departamento: (pymysql.err.OperationalError) (1049, "Unknown database 'none'")
(Background on this error at: https://sqlalche.me/e/20/e3q8)

Transformando: stg_programa -> Programa

[1] Leyendo datos desde stg_programa...

[ERROR] Fallo en transformación de stg_programa: (pymysql.err.OperationalError) (1049, "Unknown database 'none'")
(Background on this error at: https://sqlalche.me/e/20/e3q8)

Transformando: stg_curso -> Curso

[1] Leyendo datos desde stg_curso...

[ERROR] Fallo en transformación de stg_curso: (pymysql.err.OperationalError) (1049, "Unknown database 'none'")
(Background on this error at: https://sqlalche.me/e/20/e3q8)


2026-05-02 00:20:09,812 - ERROR - Error en stg_curso_programa: (pymysql.err.OperationalError) (1049, "Unknown database 'none'")
(Background on this error at: https://sqlalche.me/e/20/e3q8)
Traceback (most recent call last):
  File "c:\Users\maria\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sqlalchemy\engine\base.py", line 143, in __init__
    self._dbapi_connection = engine.raw_connection()
                             ~~~~~~~~~~~~~~~~~~~~~^^
  File "c:\Users\maria\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sqlalchemy\engine\base.py", line 3317, in raw_connection
    return self.pool.connect()
           ~~~~~~~~~~~~~~~~~^^
  File "c:\Users\maria\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sqlalchemy\pool\base.py", line 448, in connect
    return _ConnectionFairy._checkout(self)
           ~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^
  File "c:\Users\maria\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sqlalchemy\pool\base.py", line 1272, in _c


Transformando: stg_curso_programa -> CursoPrograma

[1] Leyendo datos desde stg_curso_programa...

[ERROR] Fallo en transformación de stg_curso_programa: (pymysql.err.OperationalError) (1049, "Unknown database 'none'")
(Background on this error at: https://sqlalche.me/e/20/e3q8)

Transformando: stg_docente -> Docente

[1] Leyendo datos desde stg_docente...

[ERROR] Fallo en transformación de stg_docente: (pymysql.err.OperationalError) (1049, "Unknown database 'none'")
(Background on this error at: https://sqlalche.me/e/20/e3q8)

Transformando: stg_estudiante -> Estudiante

[1] Leyendo datos desde stg_estudiante...

[ERROR] Fallo en transformación de stg_estudiante: (pymysql.err.OperationalError) (1049, "Unknown database 'none'")
(Background on this error at: https://sqlalche.me/e/20/e3q8)

Transformando: stg_dictado -> Dictado

[1] Leyendo datos desde stg_dictado...

[ERROR] Fallo en transformación de stg_dictado: (pymysql.err.OperationalError) (1049, "Unknown database 'none'")
(Backg

2026-05-02 00:20:09,995 - ERROR - Error en stg_dictado: (pymysql.err.OperationalError) (1049, "Unknown database 'none'")
(Background on this error at: https://sqlalche.me/e/20/e3q8)
Traceback (most recent call last):
  File "c:\Users\maria\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sqlalchemy\engine\base.py", line 143, in __init__
    self._dbapi_connection = engine.raw_connection()
                             ~~~~~~~~~~~~~~~~~~~~~^^
  File "c:\Users\maria\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sqlalchemy\engine\base.py", line 3317, in raw_connection
    return self.pool.connect()
           ~~~~~~~~~~~~~~~~~^^
  File "c:\Users\maria\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sqlalchemy\pool\base.py", line 448, in connect
    return _ConnectionFairy._checkout(self)
           ~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^
  File "c:\Users\maria\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sqlalchemy\pool\base.py", line 1272, in _checkout


Transformando: stg_inscripcion -> Inscripcion

[1] Leyendo datos desde stg_inscripcion...

[ERROR] Fallo en transformación de stg_inscripcion: (pymysql.err.OperationalError) (1049, "Unknown database 'none'")
(Background on this error at: https://sqlalche.me/e/20/e3q8)

Transformando: stg_examen -> Examen

[1] Leyendo datos desde stg_examen...

[ERROR] Fallo en transformación de stg_examen: (pymysql.err.OperationalError) (1049, "Unknown database 'none'")
(Background on this error at: https://sqlalche.me/e/20/e3q8)

Transformando: stg_evaluacion_curso -> EvaluacionCurso

[1] Leyendo datos desde stg_evaluacion_curso...

[ERROR] Fallo en transformación de stg_evaluacion_curso: (pymysql.err.OperationalError) (1049, "Unknown database 'none'")
(Background on this error at: https://sqlalche.me/e/20/e3q8)

TRANSFORMACIÓN COMPLETADA


In [ ]:
# ╔════════════════════════════════════════════════════════════════════════════╗
# ║            CELDA 7: REPORTE FINAL Y ESTADÍSTICAS CONSOLIDADAS              ║
# ╚════════════════════════════════════════════════════════════════════════════╝

# PROPÓSITO:
# Generar RESUMEN EJECUTIVO de la transformación completada.
# Mostrar para cada tabla: estadísticas de cada etapa del proceso.
#
# INFORMACIÓN MOSTRADA:
# - Lectura: registros extraídos de STAGING
# - Transformación: válidos/rechazados/duplicados
# - Inserción: registros cargados en DWH
# - Resultado: éxito/error de cada tabla

# ════════════════════════════════════════════════════════════════════════════

print("="*80)
print("📋 REPORTE CONSOLIDADO DE TRANSFORMACIÓN")
print("="*80)

# Acumuladores globales
total_lectura = 0
total_validos = 0
total_rechazados = 0
total_duplicados = 0
total_insertados = 0
total_errores_insert = 0
tablas_ok = 0
tablas_error = 0

# Procesar cada tabla y acumular
for tabla, stats in reporte_general.items():
    print(f"\n{'─'*80}")
    print(f"📊 {tabla}")
    print(f"{'─'*80}")
    
    if 'error' in stats:
        # Tabla con ERROR
        print(f"  [❌ ERROR] {stats['error']}")
        tablas_error += 1
        
    else:
        # Tabla OK - mostrar detalles
        lectura = stats['lectura']
        transformacion = stats['transformacion']
        insercion = stats['insercion']
        final = stats['final']
        
        # Acumular globales
        total_lectura += lectura
        total_validos += transformacion['válidos']
        total_rechazados += transformacion['rechazados']
        total_duplicados += transformacion['duplicados']
        total_insertados += insercion['insertados']
        total_errores_insert += insercion['errores']
        tablas_ok += 1
        
        # Mostrar etapas
        print(f"\n  LECTURA (desde STG):")
        print(f"    → Registros extraídos: {lectura:8}")
        
        print(f"\n  TRANSFORMACIÓN (limpieza/validación):")
        print(f"    → Total procesado:     {transformacion['total']:8}")
        print(f"    → Válidos:             {transformacion['válidos']:8} ✓")
        print(f"    → Rechazados:          {transformacion['rechazados']:8} ✗")
        print(f"    → Duplicados removidos:{transformacion['duplicados']:8}")
        
        # Calcular porcentaje de rechazo
        if transformacion['total'] > 0:
            pct_rechazo = (transformacion['rechazados'] / transformacion['total']) * 100
            print(f"    → Tasa de rechazo:     {pct_rechazo:8.1f}%")
        
        print(f"\n  INSERCIÓN (carga en DWH):")
        print(f"    → Insertados:          {insercion['insertados']:8} ✓")
        print(f"    → Errores:             {insercion['errores']:8} ✗")
        print(f"    → Lotes procesados:    {insercion['lotes']:8}")
        
        print(f"\n  VALIDACIÓN POST-INSERCIÓN:")
        if final == insercion['insertados']:
            print(f"    → [✓ OK] Registros en BD: {final:8} (coinciden)")
        else:
            print(f"    → [⚠️  WARN] Registros en BD: {final:8} (inserción: {insercion['insertados']})")

# ════════════════════════════════════════════════════════════════════════════
# RESUMEN GLOBAL
# ════════════════════════════════════════════════════════════════════════════

print("\n" + "="*80)
print("🎯 RESUMEN GLOBAL")
print("="*80)

print(f"\nTABLAS PROCESADAS:")
print(f"  ✓ Exitosas: {tablas_ok:3}")
print(f"  ✗ Con error: {tablas_error:3}")
print(f"  TOTAL:       {tablas_ok + tablas_error:3}")

print(f"\nDATOS PROCESADOS (GLOBAL):")
print(f"  → Leídos desde STAGING:  {total_lectura:8} registros")
print(f"  → Válidos después transform: {total_validos:8} registros")
print(f"  → Rechazados:            {total_rechazados:8} registros")
print(f"  → Duplicados removidos:  {total_duplicados:8} registros")
print(f"  → Insertados en DWH:     {total_insertados:8} registros")
print(f"  → Errores en inserción:  {total_errores_insert:8} registros")

if total_lectura > 0:
    pct_rechazo_global = (total_rechazados / total_lectura) * 100
    print(f"\n  TASA DE RECHAZO GLOBAL: {pct_rechazo_global:.1f}%")

if total_insertados > 0:
    print(f"\n  ✓ RESULTADO: {total_insertados} registros exitosamente cargados en DWH")
else:
    print(f"\n  ⚠️  ADVERTENCIA: No se insertó ningún registro")

LoggerManager.info(f"RESUMEN FINAL: {total_insertados} registros insertados, {total_rechazados} rechazados")

print("\n" + "="*80)
print("✓ FIN DE TRANSFORMACIÓN")
print("="*80 + "\n")


2026-05-02 00:20:10,324 - INFO - Transformación completada exitosamente



REPORTE DE TRANSFORMACIÓN

[ERROR] stg_facultad: (pymysql.err.OperationalError) (1049, "Unknown database 'none'")
(Background on this error at: https://sqlalche.me/e/20/e3q8)

[ERROR] stg_departamento: (pymysql.err.OperationalError) (1049, "Unknown database 'none'")
(Background on this error at: https://sqlalche.me/e/20/e3q8)

[ERROR] stg_programa: (pymysql.err.OperationalError) (1049, "Unknown database 'none'")
(Background on this error at: https://sqlalche.me/e/20/e3q8)

[ERROR] stg_curso: (pymysql.err.OperationalError) (1049, "Unknown database 'none'")
(Background on this error at: https://sqlalche.me/e/20/e3q8)

[ERROR] stg_curso_programa: (pymysql.err.OperationalError) (1049, "Unknown database 'none'")
(Background on this error at: https://sqlalche.me/e/20/e3q8)

[ERROR] stg_docente: (pymysql.err.OperationalError) (1049, "Unknown database 'none'")
(Background on this error at: https://sqlalche.me/e/20/e3q8)

[ERROR] stg_estudiante: (pymysql.err.OperationalError) (1049, "Unknown d